<a href="https://colab.research.google.com/github/pablonvsx/pisi3-ufrpe/blob/main/data-science/notebooks/ML/analise_final/CONCLUS%C3%83O_GERAL_DA_INVESTIGA%C3%87%C3%83O.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise Geral dos Experimentos e Evolução da Estratégia de Classificação

## Contextualização

Os experimentos iniciais deste projeto tinham como objetivo desenvolver modelos de Machine Learning capazes de classificar a qualidade da água em quatro categorias distintas:

```text
Excelente
Boa
Atenção
Crítica
```

Para isso, foram avaliados diferentes algoritmos de classificação supervisionada, incluindo Random Forest, LightGBM e SVM. Entretanto, durante as primeiras avaliações dos modelos, foi identificado um comportamento recorrente que motivou uma investigação mais aprofundada sobre a estrutura dos dados e do próprio problema de classificação.

## Identificação do Problema Inicial

As primeiras execuções apresentaram um padrão consistente nas métricas da classe **Crítica**.

Em diversos experimentos observou-se:

* Recall extremamente elevado;
* Precisão extremamente baixa;
* Grande quantidade de falsos positivos;
* Instabilidade entre diferentes algoritmos.

Em alguns cenários, a classe Crítica apresentava recalls superiores a 60%, enquanto a precisão permanecia abaixo de 10%.

Esse comportamento indicava que os modelos estavam classificando um grande número de amostras como Crítica, mas grande parte dessas previsões estava incorreta.

Inicialmente, a hipótese considerada foi a existência de um forte desbalanceamento entre as classes.

## Testes de Balanceamento

A primeira estratégia adotada consistiu na aplicação de técnicas de balanceamento das classes.

Foram realizados testes utilizando:

* `class_weight="balanced"`;
* Pesos manuais definidos experimentalmente;
* Diferentes combinações de pesos para favorecer as classes minoritárias.

Os resultados mostraram que o balanceamento realmente aumentava a capacidade dos modelos em identificar amostras críticas. Entretanto, esse ganho ocorria principalmente através do aumento do recall, enquanto a precisão permanecia baixa.

Na prática, o modelo passava a identificar mais amostras críticas, mas ao custo de gerar uma quantidade excessiva de falsos positivos.

Isso indicava que o desbalanceamento não era a única causa do problema.

## Investigação da Sobreposição Entre Classes

A etapa seguinte consistiu na análise das matrizes de confusão e dos relatórios de classificação.

Foi observado que a maior parte dos erros não ocorria entre classes extremas, como Excelente e Crítica, mas sim entre as categorias:

```text
Atenção
Crítica
```

Independentemente do algoritmo utilizado, os modelos apresentavam dificuldade em distinguir essas duas classes.

Análises exploratórias adicionais mostraram que ambas possuíam características ambientais muito semelhantes, apresentando forte sobreposição nos dados.

Essa observação foi reforçada pelos resultados obtidos durante a clusterização, onde as amostras pertencentes a essas categorias ocupavam regiões muito próximas no espaço de atributos.

Na prática, os modelos estavam tentando aprender uma separação que os próprios dados não apresentavam de forma clara.

## Investigação Temporal

Uma nova hipótese levantada foi a possível influência da distribuição temporal das amostras.

Foi realizada uma análise da frequência das classes ao longo dos anos, identificando-se que os registros classificados como Crítica estavam concentrados em períodos específicos da série histórica.

Os resultados mostraram que o intervalo entre **2000 e 2008** possuía a maior representatividade da classe Crítica, tornando-se o recorte temporal mais adequado para novos experimentos.

A partir dessa descoberta, foi criado um novo conjunto de dados contendo apenas esse período.

## Resultados com o Recorte Temporal

Os experimentos realizados utilizando exclusivamente o intervalo de 2000 a 2008 apresentaram melhorias importantes.

Houve:

* Aumento da representatividade das classes minoritárias;
* Redução do desequilíbrio observado no conjunto original;
* Maior estabilidade entre precisão e recall;
* Melhor desempenho geral dos modelos.

Entretanto, mesmo após essa melhoria, a principal fonte de erro continuou sendo a distinção entre as classes Atenção e Crítica.

Ou seja, o problema persistia mesmo após a correção do fator temporal.

## Revisão da Construção do Rótulo

Diante desse cenário, foi realizada uma análise detalhada da forma como o rótulo havia sido construído.

O rótulo original era gerado a partir de um score ambiental baseado em cinco parâmetros físico-químicos:

* pH;
* Oxigênio dissolvido;
* DBO;
* Nitrato;
* Amônia.

A partir desse score, as classes eram definidas da seguinte forma:

```text
5 → Excelente
4 → Boa
3 → Atenção
0, 1 ou 2 → Crítica
```

Durante a análise, observou-se que a diferença entre Atenção e Crítica frequentemente dependia da condição de apenas um parâmetro.

Isso fazia com que amostras muito semelhantes fossem colocadas em categorias diferentes, criando uma fronteira extremamente difícil de ser aprendida pelos modelos.

Além disso, para evitar problemas de *data leakage*, os parâmetros utilizados na construção do rótulo não eram utilizados diretamente como variáveis de entrada dos algoritmos, tornando a tarefa ainda mais complexa.

## Reformulação do Problema

Após diversos testes e análises, concluiu-se que a principal limitação não estava associada aos algoritmos, aos hiperparâmetros ou às estratégias de balanceamento.

A principal limitação estava na própria definição do problema.

Por esse motivo, optou-se por reformular o rótulo em três classes:

```text
Adequada
Boa
Não adequada
```

onde:

```text
Score 5 → Adequada
Score 4 → Boa
Score 0, 1, 2 ou 3 → Não adequada
```

Essa nova estrutura elimina a separação artificial entre Atenção e Crítica, agrupando ambas em uma única categoria que representa situações que exigem acompanhamento técnico.

## Justificativa para o Novo Rótulo

A decisão também está alinhada ao objetivo prático do AquaSense.

O sistema não tem como finalidade substituir uma análise laboratorial ou uma avaliação realizada por especialistas ambientais.

Seu papel é atuar como uma ferramenta de apoio à tomada de decisão.

Na prática, o aplicativo recebe:

* Observações visuais;
* Informações coletadas por colaboradores;
* Medições simplificadas realizadas em campo.

Com base nesses dados, o modelo realiza uma classificação preliminar da condição do corpo hídrico.

Dessa forma, é mais coerente que o algoritmo responda:

```text
Adequada
Boa
Não adequada
```

do que tentar afirmar com precisão se uma água está em condição Atenção ou Crítica sem a presença de uma validação técnica especializada.

Quando uma amostra é classificada como **Não adequada**, ela passa a representar um indicativo de possível comprometimento ambiental, sinalizando ao gestor a necessidade de investigação complementar.

## Conclusão Geral

Os resultados obtidos ao longo dos experimentos demonstraram que o principal desafio do projeto não estava relacionado à escolha do algoritmo, mas sim à estrutura do problema de classificação.

A investigação permitiu identificar:

* Sobreposição significativa entre Atenção e Crítica;
* Influência da distribuição temporal das amostras;
* Limitações na definição original do rótulo;
* Benefícios da simplificação do problema para três classes.

A reformulação adotada tornou o problema mais coerente com os dados disponíveis e mais compatível com a finalidade prática do AquaSense.

Assim, o modelo passa a atuar como uma ferramenta de triagem inteligente, auxiliando gestores ambientais na identificação de situações que demandam monitoramento ou intervenção, sem substituir o papel da avaliação técnica especializada.
